# Горизонт передбачуваності в моделі Лоренца

**Автори:** Артем, Еван

## Фабула

У 1961 році Едвард Лоренц запустив спрощену модель атмосферної конвекції на комп'ютері LGP-30 і виявив, що два запуски з майже однаковими початковими умовами (відмінність у п'ятому десятковому знаку) через короткий час дають **принципово різні** прогнози. Так народилася теорія детермінованого хаосу і фраза «ефект метелика».

Це відкриття має конкретний практичний наслідок: **існує фундаментальна межа передбачуваності погоди** — не через обмеження комп'ютерів, а через саму природу динамічних рівнянь. Сьогодні ця межа складає 10–14 днів, і жодні покращення точності вимірювань її радикально не відсунуть.

У цьому проєкті ми дослідимо механізм цього явища на класичній системі Лоренца, поєднуючи **символьний аналіз стійкості** (локальна картина — нерухомі точки та власні значення) з **чисельним експериментом** (глобальна картина — розходження траєкторій у часі).

## Математична модель

Система Лоренца:
$$
\begin{cases}
\dot x = \sigma(y - x) \\
\dot y = x(\rho - z) - y \\
\dot z = xy - \beta z
\end{cases}
$$

Класичні параметри, для яких спостерігається хаос:
$$\sigma = 10, \quad \rho = 28, \quad \beta = 8/3.$$

Фізичний зміст змінних: $x$ — інтенсивність конвекції, $y$ — різниця температур між висхідним і низхідним потоками, $z$ — відхилення вертикального температурного профілю від лінійного.

## План дослідження

1. **Символьно** знайти всі нерухомі точки системи та визначити їхній тип через власні значення лінеаризації.
2. **Чисельно** проінтегрувати дві траєкторії з близькими початковими умовами та виміряти, як росте відстань між ними.
3. **Оцінити показник Ляпунова** $\lambda$ через лінійну регресію $\ln d(t)$ і порівняти з теоретичним значенням.
4. **Візуалізувати** результати інтерактивно (Plotly): атрактор у 3D з нанесеними нерухомими точками + семилогарифмічний графік розходження.

In [2]:
import numpy as np
from scipy.integrate import solve_ivp
import json

import plotly.graph_objects as go



Частина 2. Чисельна симуляція чутливості до початкових умов
Тепер запустимо дві траєкторії з мікроскопічно різними початковими умовами (різниця $10^{-6}$ у координаті $x$, тобто приблизно шум при 6-значній точності вимірювання). Для високоточного інтегрування використаємо метод Дорманда–Принса 8-го порядку (DOP853) з відносною точністю $10^{-10}$ — це набагато менше, ніж наше збурення, тож спостережуване розходження буде справжнім, а не числовим артефактом.

In [3]:
# Параметри як звичайні float для numpy/scipy
SIGMA, RHO, BETA = 10.0, 28.0, 8.0 / 3.0

def lorenz(t, state):
    xv, yv, zv = state
    return [
        SIGMA * (yv - xv),
        xv * (RHO - zv) - yv,
        xv * yv - BETA * zv,
    ]

# Прогрів: інтегруємо від [1, 1, 1] протягом 10 одиниць часу,
# щоб вийти на сам атрактор і позбутися перехідного процесу
burn = solve_ivp(lorenz, (0.0, 10.0), [1.0, 1.0, 1.0],
                 method="DOP853", rtol=1e-10, atol=1e-12)
p0 = burn.y[:, -1]
print(f"Точка на атракторі: ({p0[0]:.3f}, {p0[1]:.3f}, {p0[2]:.3f})")

# Дві близькі початкові умови
ic_A = p0.copy()
ic_B = p0 + np.array([1e-6, 0.0, 0.0])

# Основна симуляція на [0, 25]
T_MAX = 25.0
t_eval = np.linspace(0.0, T_MAX, 5001)
opts = dict(method="DOP853", rtol=1e-10, atol=1e-12, t_eval=t_eval)

sol_A = solve_ivp(lorenz, (0.0, T_MAX), ic_A, **opts)
sol_B = solve_ivp(lorenz, (0.0, T_MAX), ic_B, **opts)

print(f"Початкове збурення: d(0) = {np.linalg.norm(ic_A - ic_B):.1e}")

Точка на атракторі: (-4.903, -3.744, 24.691)
Початкове збурення: d(0) = 1.0e-06


## Частина 1. Символьний аналіз стійкості нерухомих точок

Щоб зрозуміти, чому траєкторії розходяться, подивимось спочатку на локальну структуру поля навколо нерухомих точок. Нерухома точка $\mathbf{p}^*$ — це стан, де $\dot{\mathbf{p}} = \mathbf{0}$. Поведінка траєкторій поблизу визначається власними значеннями матриці Якобі $J(\mathbf{p}^*)$:

- усі $\mathrm{Re}(\lambda_i) < 0$ → стійка точка, траєкторії її притягують;
- хоча б одне $\mathrm{Re}(\lambda_i) > 0$ → нестійка точка, траєкторії відштовхуються у відповідному напрямку.

Зараз ми покажемо ключовий факт: **усі нерухомі точки системи Лоренца при $\rho = 28$ нестійкі**. Це означає, що траєкторії ніде не зупиняються, а весь час «блукають» між нестабільними точками — звідси й хаотична поведінка.

In [4]:
# Оголошуємо символьні змінні
var('x y z')

# Параметри: 8/3 у Sage — точний раціональний, не float
sigma_val, rho_val, beta_val = 10, 28, 8/3

# Права частина системи — просто список символьних виразів
F = [
    sigma_val * (y - x),
    x * (rho_val - z) - y,
    x*y - beta_val*z,
]

# jacobian() у Sage — це глобальна функція, а не метод
J = jacobian(F, [x, y, z])

print("Матриця Якобі J(x, y, z):")
show(J)

Матриця Якобі J(x, y, z):


[    -10      10       0]
[-z + 28      -1      -x]
[      y       x    -8/3]

### 1.1. Знаходимо нерухомі точки

Розв'язуємо систему $F(x, y, z) = \mathbf{0}$ точно, через `solve()`:

In [5]:
fixed_points = solve(
    [F[0] == 0, F[1] == 0, F[2] == 0],
    [x, y, z],
    solution_dict=True
)

print(f"Знайдено нерухомих точок: {len(fixed_points)}")
for i, pt in enumerate(fixed_points, 1):
    print(f"  P{i}: x = {pt[x]}, y = {pt[y]}, z = {pt[z]}")

Знайдено нерухомих точок: 3
  P1: x = 6*sqrt(2), y = 6*sqrt(2), z = 27
  P2: x = -6*sqrt(2), y = -6*sqrt(2), z = 27
  P3: x = 0, y = 0, z = 0


Отримали три точки:
- **$O = (0, 0, 0)$** — початок координат, відповідає стаціонарному стану без конвекції;
- **$C^\pm = (\pm 6\sqrt{2},\ \pm 6\sqrt{2},\ 27)$** — два симетричні центри «крил метелика», відповідають двом режимам стаціонарної конвекції (за або проти годинникової стрілки).

### 1.2. Класифікація нерухомих точок за власними значеннями

Обчислюємо $J$ у кожній точці, знаходимо спектр і класифікуємо.

In [6]:
def classify(eigenvalues):
    """Класифікує нерухому точку за списком комплексних власних значень."""
    real_parts = [CDF(e).real() for e in eigenvalues]
    has_positive = any(r > 1e-9 for r in real_parts)
    has_negative = any(r < -1e-9 for r in real_parts)
    any_complex = any(abs(CDF(e).imag()) > 1e-9 for e in eigenvalues)

    if has_positive and has_negative:
        base = "седло"
    elif has_positive:
        base = "нестійке джерело"
    elif has_negative:
        base = "стійкий стік"
    else:
        base = "центр"

    if any_complex:
        base += " (з обертанням)"
    return base


def fmt_eig(e):
    """Красиве форматування комплексного власного значення."""
    c = CDF(e)
    re, im = float(c.real()), float(c.imag())
    if abs(im) < 1e-9:
        return f"{re:+.3f}"
    return f"{re:+.3f} {'+' if im >= 0 else '-'} {abs(im):.3f}i"


# Збираємо дані для зведеної таблиці
table_rows = [["Точка", "Координати", "Власні значення Якобіана", "Тип"]]
fixed_numeric = []   # збережемо для візуалізації

for i, pt in enumerate(fixed_points):
    label = "O" if i == 0 else f"C{'+' if i == 1 else '-'}"
    coords = (float(pt[x]), float(pt[y]), float(pt[z]))
    fixed_numeric.append((label, coords))

    J_at = J.subs(pt).change_ring(CDF)
    eigs = J_at.eigenvalues()

    table_rows.append([
        label,
        f"({coords[0]:.2f}, {coords[1]:.2f}, {coords[2]:.2f})",
        ",  ".join(fmt_eig(e) for e in eigs),
        classify(eigs),
    ])

show(table(table_rows, header_row=True, frame=True))

Точка,Координати,Власні значення Якобіана,Тип
O,"(8.49, 8.49, 27.00)","-13.855, +0.094 + 10.195i, +0.094 - 10.195i",седло (з обертанням)
C+,"(-8.49, -8.49, 27.00)","-13.855, +0.094 + 10.195i, +0.094 - 10.195i",седло (з обертанням)
C-,"(0.00, 0.00, 0.00)","-22.828, +11.828, -2.667",седло


**Спостереження.** Усі три точки нестійкі:

- В **$O$** — типове седло: одна координата притягує, дві відштовхують. Траєкторія не може зависнути у нулі.
- В **$C^\pm$** — комплексні власні значення з **додатною** дійсною частиною ($\approx +0.09$). Це нестійкий фокус: траєкторія робить розкручуючу спіраль навколо точки, поки не «перескочить» на інше крило.

Оскільки притягувальних точок не існує, траєкторія приречена вічно блукати між $C^+$ і $C^-$ — ми побачимо це у 3D-візуалізації. Далі ми покажемо чисельно, що таке блукання супроводжується експоненційним розходженням близьких траєкторій.